# AgriCL-Net, step 6: edge models (offline, Raspberry Pi)

Distils two small **student** models from the accurate **teachers**, exports them to ONNX and quantizes them to
**INT8**, then measures size, speed and accuracy before and after:

| Task | Teacher | Student |
| --- | --- | --- |
| Lesion segmentation | SAM2-UNet (notebook 01) | LinkNet-ResNet18 (about 11.5 M parameters) |
| Disease classification | Swin-T (notebook 03 or 04) | MobileNetV3-Large |

The result is an **edge bundle** that runs with ONNX Runtime + NumPy only (no PyTorch), with the colour-index
leaf mask. The app also accepts it, so it can run fully offline.

**Needs:** finished runs of notebook 01 and of notebook 03 or 04, and a T4 GPU. Run once with `SMOKE_TEST = True`,
then set it to `False` and run all. Both trainings resume after a disconnect.

## Settings

In [ ]:
SIZE = 'l'               # lesion teacher (notebook 01)
IMG_SIZE = 352
MODEL = 'swin_tiny_patch4_window7_224.ms_in22k_ft_in1k'
TEACHER = 'continual'    # classifier teacher: 'continual' (notebook 04, replay_kd) or 'joint' (notebook 03)
CLS_EPOCHS = 30
SEG_EPOCHS = 40
SMOKE_TEST = True        # True: 40 images per split, 1 epoch each. Set False for the real run.

## GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Storage and project code from GitHub

Works on **Colab** and **Kaggle**:
* Colab: everything is kept in Google Drive, `MyDrive/Colab Notebooks/agricl-net`.
* Kaggle: turn on *Settings > Accelerator > GPU T4* and *Settings > Internet*. Results go to
  `/kaggle/working/agricl-net` and are kept when you *Save Version*. Downloads go to `/tmp` (fetched again each
  session). To use an earlier notebook's results (or resume a run), attach that notebook's output with
  *Add Input > Notebook Output*: it is copied in automatically below.

In [ ]:
import os, sys, glob, shutil
ON_KAGGLE = os.path.exists('/kaggle/working')
if ON_KAGGLE:
    WORK = '/tmp'                               # scratch space: code, unzipped data
    DRIVE = '/kaggle/working/agricl-net'        # results, saved with each notebook version
    CACHE = '/tmp/agricl-cache'                 # dataset zip and SAM2 checkpoint
    for src in glob.glob('/kaggle/input/**/agricl-net/runs/*', recursive=True):
        dst = f'{DRIVE}/runs/{os.path.basename(src)}'
        if os.path.isdir(src) and not os.path.exists(dst):
            shutil.copytree(src, dst)
            print('copied earlier results:', os.path.basename(src))
else:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content'
    DRIVE = '/content/drive/MyDrive/Colab Notebooks/agricl-net'   # dataset zip, SAM2 checkpoint and all results
    CACHE = f'{DRIVE}/cache'
os.makedirs(CACHE, exist_ok=True)
os.makedirs(f'{DRIVE}/runs', exist_ok=True)

CODE = f'{WORK}/AGRICL-NET'
if os.path.exists(CODE):
    !git -C {CODE} pull -q
else:
    !git clone -q https://github.com/Dev02005/AGRICL-NET.git {CODE}
sys.path.insert(0, CODE)
!git -C {CODE} log -1 --format='code version: %h %s'

## PlantSeg v7 dataset (1.1 GB; on Colab downloaded to Drive once)

In [ ]:
import hashlib
ZIP = f'{CACHE}/plantseg.zip'
if not os.path.exists(ZIP):
    !wget -q --show-progress -O "{ZIP}" "https://zenodo.org/records/17719108/files/plantseg.zip?download=1"

h = hashlib.md5()
with open(ZIP, 'rb') as f:
    for chunk in iter(lambda: f.read(1 << 24), b''):
        h.update(chunk)
assert h.hexdigest() == '9358a66dff88cdd15c4fe009763c40a3', f'bad download, md5 {h.hexdigest()}: delete {ZIP} and rerun'

DATA = f'{WORK}/data/plantseg'
if not os.path.exists(f'{DATA}/Metadata.csv'):
    !unzip -q "{ZIP}" -d "{WORK}/data"

## SAM2-UNet code and SAM2 checkpoint

In [ ]:
!pip -q install hydra-core==1.3.2
SAM2UNET = f'{WORK}/SAM2-UNet'
if not os.path.exists(SAM2UNET):
    !git clone -q https://github.com/WZH0120/SAM2-UNet.git {SAM2UNET}
    !git -C {SAM2UNET} checkout -q 01598e5e9912ffb23f965ecbebf4d1dfecbaa56e

from agricl.models.sam2unet import CHECKPOINTS, CHECKPOINT_URL
CKPT = f'{CACHE}/{CHECKPOINTS[SIZE]}'
if not os.path.exists(CKPT):
    !wget -q --show-progress -O "{CKPT}" "{CHECKPOINT_URL}{CHECKPOINTS[SIZE]}"
print(CKPT, os.path.getsize(CKPT) // 2**20, 'MB')

In [ ]:
!pip -q install timm onnx onnxruntime
import json, shutil
RUN = f'edge_{TEACHER}' + ('_smoke' if SMOKE_TEST else '')
OUT = f'{DRIVE}/runs/{RUN}'
os.makedirs(OUT, exist_ok=True)
IMG_CACHE = f'{WORK}/plantseg_640'
short = MODEL.split('.')[0]

lesion_teacher = f"{DRIVE}/runs/lesion_sam2unet_{SIZE.replace('+', 'p')}_{IMG_SIZE}/best.pt"
assert os.path.exists(lesion_teacher), 'run notebook 01 (SMOKE_TEST = False) first'
if TEACHER == 'joint':
    run = f'{DRIVE}/runs/classifier_{short}'
    teacher_weights, teacher_classes, kind = f'{run}/best.pt', f'{run}/classes.json', 'timm'
else:
    run = f'{DRIVE}/runs/continual_{short}'
    teacher_weights, kind = f'{run}/replay_kd_final.pt', 'incremental'
    teacher_classes = f'{OUT}/teacher_classes.json'
    json.dump([d for r in json.load(open(f'{run}/rounds.json')) for d in r['diseases']], open(teacher_classes, 'w'))
assert os.path.exists(teacher_weights), f'{teacher_weights} not found: run notebook 03 or 04 first'
print('results ->', OUT)

## Distil the classifier student (MobileNetV3)

In [ ]:
from agricl.edge_train import EdgeConfig, distill_classifier, distill_lesion

cfg = EdgeConfig(
    data_root=DATA, out_dir=OUT, cache_dir=IMG_CACHE,
    sam2unet_repo=SAM2UNET, sam2_size=SIZE, sam2_checkpoint=CKPT, lesion_teacher=lesion_teacher, lesion_size=IMG_SIZE,
    classifier_teacher_kind=kind, classifier_teacher_weights=teacher_weights,
    classifier_teacher_classes=teacher_classes, teacher_model_name=MODEL,
    cls_epochs=1 if SMOKE_TEST else CLS_EPOCHS, seg_epochs=1 if SMOKE_TEST else SEG_EPOCHS,
    limit=40 if SMOKE_TEST else 0,
)
cls_results = distill_classifier(cfg)

## Distil the lesion student (LinkNet-ResNet18)

In [ ]:
seg_results = distill_lesion(cfg)

## Export to ONNX, quantize to INT8, and compare size, speed and accuracy

In [ ]:
import random, numpy as np, pandas as pd, torch
from agricl.edge import OnnxModel
from agricl.edge_eval import (classifier_inputs, classifier_top1, file_mb, latency_ms, lesion_inputs,
                              lesion_scores, quantize_int8)
from agricl.edge_train import export_onnx, splits
from agricl.models.classifier import build_classifier
from agricl.models.linknet import LinkNet18

data = splits(cfg)
EDGE = f'{OUT}/edge_bundle'
os.makedirs(EDGE, exist_ok=True)
classes = json.load(open(f'{OUT}/classifier/classes.json'))
c2i = {d: i for i, d in enumerate(classes)}

clf = build_classifier(len(classes), cfg.student_classifier, pretrained=False)
clf.load_state_dict(torch.load(f'{OUT}/classifier/best.pt', map_location='cpu', weights_only=True))
seg = LinkNet18(pretrained=False)
seg.load_state_dict(torch.load(f'{OUT}/lesion/best.pt', map_location='cpu', weights_only=True))
export_onnx(clf, f'{EDGE}/classifier_fp32.onnx', 224)
export_onnx(seg, f'{EDGE}/lesion_fp32.onnx', IMG_SIZE)

calib = random.Random(0).sample(data['train'], min(100, len(data['train'])))
quantize_int8(f'{EDGE}/classifier_fp32.onnx', f'{EDGE}/classifier_int8.onnx', classifier_inputs(calib, 224))
quantize_int8(f'{EDGE}/lesion_fp32.onnx', f'{EDGE}/lesion_int8.onnx', lesion_inputs(calib, IMG_SIZE))

shutil.copy(f'{OUT}/classifier/classes.json', f'{EDGE}/classes.json')
shutil.copy(f'{OUT}/classifier/calibration.json', f'{EDGE}/calibration.json')
json.dump({'lesion': {'onnx': 'lesion_int8.onnx', 'img_size': IMG_SIZE},
           'classifier': {'onnx': 'classifier_int8.onnx', 'img_size': 224,
                          'classes': 'classes.json', 'calibration': 'calibration.json'}},
          open(f'{EDGE}/edge_bundle.json', 'w'), indent=2)

test = random.Random(1).sample(data['test_seen'], min(300, len(data['test_seen'])))
cls_x, seg_x = classifier_inputs(test, 224), lesion_inputs(test, IMG_SIZE)
labels = np.array([c2i[r.disease] for r in test])
rows = []
for task, prec in [('classifier', 'fp32'), ('classifier', 'int8'), ('lesion', 'fp32'), ('lesion', 'int8')]:
    path = f'{EDGE}/{task}_{prec}.onnx'
    m1, mall = OnnxModel(path, threads=1), OnnxModel(path)
    x = cls_x if task == 'classifier' else seg_x
    acc = (classifier_top1(mall, x, labels) if task == 'classifier'
           else lesion_scores(mall, test, x)['mean_dice'])
    rows.append({'model': task, 'precision': prec, 'size_mb': round(file_mb(path), 1),
                 'ms_1_thread': round(latency_ms(m1, x[0]), 1), 'ms_all_threads': round(latency_ms(mall, x[0]), 1),
                 'accuracy': round(acc, 4), 'metric': 'top-1' if task == 'classifier' else 'mean Dice'})
table = pd.DataFrame(rows)
table.to_csv(f'{OUT}/edge_benchmark.csv', index=False)
print(f'Colab CPU cores: {os.cpu_count()} (a Raspberry Pi 4 has 4)')
table

**How to read it:** INT8 files should be about 4x smaller and usually faster on CPU, with a small accuracy drop.
Teacher accuracy for comparison: `cls_results['teacher']` below, and notebook 01's `results.json`.

In [ ]:
print('classifier teacher vs student (test, top-1):',
      round(cls_results['teacher']['top1'], 4), 'vs', round(cls_results['student']['top1'], 4))
print('lesion student (full resolution):', json.dumps(seg_results, indent=1))

## Try the edge bundle on a few photos (ONNX Runtime only)

In [ ]:
from PIL import Image
from agricl.edge import EdgePredictor

p = EdgePredictor(EDGE)
for r in random.Random(2).sample(data['unseen'], 2) + random.Random(2).sample(data['test_seen'], 2):
    d = p.diagnose(Image.open(r.image))
    print(f"true: {r.disease:<28} -> {d.decision.status:<10} {str(d.decision.disease):<28} "
          f"severity {d.severity.percent:5.1f}% {d.severity.level:<8} {sum(d.timings.values()) * 1000:5.0f} ms")
shutil.make_archive(f'{DRIVE}/agricl_edge_bundle', 'zip', EDGE)
print(f"{DRIVE}/agricl_edge_bundle.zip", os.path.getsize(f'{DRIVE}/agricl_edge_bundle.zip') // 2**20, 'MB')

## On a Raspberry Pi
1. Raspberry Pi OS Bookworm **64-bit**, Python 3.11 to 3.14.
2. Copy the project and unzip `agricl_edge_bundle.zip` to `models/edge_bundle/`.
3. `pip install -r requirements-edge.txt`
4. `python scripts/edge_benchmark.py --bundle models/edge_bundle --images <folder of leaf photos>`
5. Or start the app: `streamlit run app/streamlit_app.py` and set the bundle folder to `models/edge_bundle`.